In [4]:
# 📦 Imports
import json, pathlib, random
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

# 📂 Setup paths
BASE = pathlib.Path.cwd()
DATA = BASE / "data"
ARTIFACTS = BASE / "artifacts"

# 📂 Load JSON helpers
def load_json_safe(path):
    try:
        return json.loads(path.read_text())
    except Exception as e:
        print(f"[warn] Could not load {path.name}: {e}")
        return {}

legacy_map = load_json_safe(DATA / "ncbi_protein_legacy_map.json")
seed_classification = load_json_safe(ARTIFACTS / "seed_classification.json")
eti_list = set(load_json_safe(ARTIFACTS / "eti_list.json"))
ivacaftor_list = set(load_json_safe(ARTIFACTS / "ivacaftor_list.json"))

# 🛠️ Feature functions
def standardize_variant(v):
    return legacy_map.get(v, v)

def get_pathogenicity(v):
    v = standardize_variant(v)
    if v in seed_classification:
        return seed_classification[v]
    if v in ivacaftor_list:
        return "responsive_ivacaftor"
    if v in eti_list:
        return "responsive_ETI"
    return "uncertain"

def pathogenic_count(v1, v2):
    def is_path(c): return c in ["pathogenic", "responsive_ivacaftor", "responsive_ETI"]
    return sum(is_path(get_pathogenicity(v)) for v in [v1, v2])

def get_modulator_group(v1, v2):
    v1s, v2s = standardize_variant(v1), standardize_variant(v2)
    vs = {v1s, v2s}
    if "F508del" in vs and len(vs) == 1: return 1
    if "F508del" in vs: return 2
    if v1s in eti_list or v2s in eti_list: return 3
    if v1s in ivacaftor_list or v2s in ivacaftor_list: return 4
    return 5

def featurize(v1, v2, age=None, sex=None):
    return {
        "pathogenic_count": pathogenic_count(v1, v2),
        "mod_group": get_modulator_group(v1, v2),
        "age": age if age is not None else -1,
        "sex_male": 1 if str(sex).upper().startswith("M") else 0
    }

def classify_cf(v1, v2):
    c1, c2 = get_pathogenicity(v1), get_pathogenicity(v2)
    is_path = lambda c: c in ["pathogenic", "responsive_ivacaftor", "responsive_ETI"]
    if is_path(c1) and is_path(c2):
        return "CF positive"
    if (is_path(c1) and c2 == "uncertain") or (is_path(c2) and c1 == "uncertain"):
        return "risk uncertain"
    return "CF negative / carrier"

# 🧬 Synthetic training data
def sample_fev1(age):
    mu = 97.0 if age < 16 else 78.3
    return max(30.0, min(120.0, random.gauss(mu, 8.0)))

def severity_from_fev1(fev1):
    return round(1.0 - (fev1 / 120.0), 3)

common_pairs = [
    ("F508del","F508del"), ("F508del","G542X"),
    ("F508del","G551D"), ("G551D","R117H"),
    ("N1303K","G542X"), ("R117H","unknown")
]

rows = []
for _ in range(2000):
    v1, v2 = random.choice(common_pairs)
    age = random.randint(6, 60)
    sex = random.choice(["M", "F"])
    fev1 = sample_fev1(age)
    sev = severity_from_fev1(fev1)
    feats = featurize(v1, v2, age, sex)
    feats.update({
        "severity_index": sev,
        "variant1": v1,
        "variant2": v2,
        "age": age,
        "sex": sex
    })
    rows.append(feats)

df = pd.DataFrame(rows)
print("✅ Synthetic dataset shape:", df.shape)
df.head()

# 🧠 Train the model
X = df[["pathogenic_count", "mod_group", "age", "sex_male"]]
y = df["severity_index"]

rf = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=10, random_state=42)
rf.fit(X, y)

print("✅ Model trained.")
print("R²:", round(r2_score(y, rf.predict(X)), 3))
print("MAE:", round(mean_absolute_error(y, rf.predict(X)), 3))

# 💾 Save model and feature order
ARTIFACTS.mkdir(exist_ok=True)

joblib.dump(rf, ARTIFACTS / "severity_model.joblib")
(ARTIFACTS / "feature_order.json").write_text(json.dumps(["pathogenic_count", "mod_group", "age", "sex_male"]))

print("✅ Saved model to:", ARTIFACTS / "severity_model.joblib")
print("✅ Saved feature order to:", ARTIFACTS / "feature_order.json")

# 🧪 Rule-based classifier tests
print("🧪 Rule-based classifier tests:")
print("F508del + F508del →", classify_cf("F508del", "F508del"))
print("F508del + R117H →", classify_cf("F508del", "R117H"))
print("R117H + unknown →", classify_cf("R117H", "unknown"))


✅ Synthetic dataset shape: (2000, 8)
✅ Model trained.
R²: 0.492
MAE: 0.051
✅ Saved model to: /Users/adithyavikram/Documents/Cystic_Fibrosis_Model/artifacts/severity_model.joblib
✅ Saved feature order to: /Users/adithyavikram/Documents/Cystic_Fibrosis_Model/artifacts/feature_order.json
🧪 Rule-based classifier tests:
F508del + F508del → CF positive
F508del + R117H → CF positive
R117H + unknown → risk uncertain
